# Fonda seed runner

Runs `scripts/run_pipeline_stages.py` at a single seed, end to end: **split generation
-> data prep -> SGD -> MLP baseline -> experience replay -> evaluation**. Every cell is
safe to re-run in a brand-new session -- clone-or-pull, dependency install, git config,
the dataset symlink and split generation are all idempotent. Just **Run All** each
session; finished work is skipped automatically and interrupted work resumes.

Writes everything under `experiments/seed_<SEED>/` and never touches the original
`experiments/` tree (seed 42) -- see `src/seed_run.py` for why that separation holds.

**One seed per session.** The disk feature cache is keyed by a hash of the split's
pixel-index array, so each seed rebuilds it from scratch regardless; running two seeds
in one session would mean holding two full caches against Kaggle's ~20GB quota at once.

**To run the next seed**, edit `SEED` in the cell below and **Run All** again in a new
session -- nothing else on this page needs to change.

**One-time setup before your first run (same as the seed-42 runner):**
1. Notebook Settings (right panel) -> **Internet: On**.
2. **Add Input** -> attach your `fonda-training-data` dataset.
3. Add-ons -> **Secrets** -> add a secret named exactly `GITHUB_TOKEN`, holding a
   GitHub Personal Access Token with **write** access to this repo (needed for
   `--git-push` -- without this, training results are lost when the session ends,
   since `/kaggle/working` does not survive a session boundary on its own).

In [ ]:
# The only thing you change between sessions. 1, 2, 3 per the multi-seed plan --
# each writes to its own experiments/seed_<SEED>/ tree, so there is no wrong order
# and no way for one seed to overwrite another's results.
SEED = 1
print(f"Running seed {SEED}")

In [ ]:
import pathlib
import subprocess

REPO_URL = "https://github.com/bartuturan/Online-Learning-for-Continuous-Forest-Monitoring-Addressing-Concept-Drift-in-Disturbance-Detection."
BRANCH = "pipeline-rerun"
ROOT = pathlib.Path("/kaggle/working/repo")

if (ROOT / ".git").exists():
    print("Repo already present -- pulling latest...")
    result = subprocess.run(["git", "pull"], cwd=ROOT, capture_output=True, text=True)
else:
    print("Cloning repo...")
    result = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(ROOT)], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)
result.check_returncode()

In [ ]:
!pip install -q "zarr>=3" "scikit-learn>=1.7"

In [ ]:
import inspect

import sklearn
import xarray as xr
import zarr
from sklearn.neural_network import MLPClassifier

print("zarr:", zarr.__version__)
print("scikit-learn:", sklearn.__version__)

engines = list(xr.backends.list_engines())
print("xarray engines:", engines)
assert "zarr" in engines, (
    "zarr isn't registered as an xarray backend -- restart the kernel (not just "
    "re-run cells) and re-run from the pip install cell above."
)

sig = inspect.signature(MLPClassifier.partial_fit)
print("MLPClassifier.partial_fit signature:", sig)
assert "sample_weight" in sig.parameters, (
    "scikit-learn is too old for sample_weight in partial_fit -- restart the "
    "kernel and re-run from the pip install cell above."
)
print("Dependency check OK.")

In [ ]:
import subprocess

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")  # Add-ons > Secrets, named exactly GITHUB_TOKEN

subprocess.run(["git", "config", "user.email", "kaggle-pipeline@users.noreply.github.com"], cwd=ROOT, check=True)
subprocess.run(["git", "config", "user.name", "Kaggle Pipeline Runner"], cwd=ROOT, check=True)

# Note: this repo's name unusually ends in a literal period -- match REPO_URL
# above exactly rather than appending ".git" (which would double it up).
push_url = f"https://{GITHUB_TOKEN}@github.com/bartuturan/Online-Learning-for-Continuous-Forest-Monitoring-Addressing-Concept-Drift-in-Disturbance-Detection."
subprocess.run(["git", "remote", "set-url", "origin", push_url], cwd=ROOT, check=True)
print("Git configured for push access (token not printed).")

In [ ]:
import pathlib

candidates = [
    pathlib.Path("/kaggle/input/fonda-training-data"),
    pathlib.Path("/kaggle/input/datasets/bartuturan/fonda-training-data"),
]
SRC = next((p for p in candidates if p.exists()), None)
if SRC is None:
    raise FileNotFoundError(
        f"fonda-training-data dataset not found under any of: {candidates}. "
        f"Attach it via 'Add Input' in the notebook's Data panel."
    )
print(f"Using dataset at: {SRC}")

# Only the feature store -- NOT data_split.npz. A seed run generates its own split
# below (make_seed_split.py); symlinking the seed-42 split here would silently
# train every seed against the same partition it's meant to vary.
name = "training_data_with_features_plus_monthly_indices.zarr"
link = ROOT / name
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(SRC / name, target_is_directory=True)
print(f"  linked {name}")

In [ ]:
import os

os.environ["MLP_FEATURE_CACHE_DIR"] = "/kaggle/working/feature_cache_disk"
print("MLP_FEATURE_CACHE_DIR set.")

### Generate this seed's split

Cheap -- one variable read (`cube_idx`), no feature computation -- so it runs here
rather than being uploaded as part of the dataset. Not committed (`*.npz` is
gitignored), so this regenerates every session; `make_seed_split.py` refuses to
overwrite an existing `data_split_seed_<N>.npz` without `--force`, which only
matters if a session is interrupted and resumed -- the split it already produced
is reused rather than silently redrawn.

In [ ]:
%cd /kaggle/working/repo
!python -u scripts/make_seed_split.py --seed {SEED}

In [ ]:
%cd /kaggle/working/repo
!python -u scripts/run_pipeline_stages.py --list --seed {SEED}

### Check the status table above before running

Confirm it matches what you expect for this seed (which stages are DONE vs TODO --
all TODO on a first run for this seed, partially DONE if resuming). The run below
executes everything still outstanding, in order: **data prep -> SGD -> MLP baseline
-> experience replay -> evaluation**, all under `experiments/seed_<SEED>/`.
Already-DONE stages are skipped instantly, so this is also the command you re-run
in a later session to continue this seed where the last one stopped.

The reservoir-sampling variants are excluded by default (they're not evaluated by
any family in `src/eval/families.py`, so skipping them costs the evaluation
nothing). Run one explicitly with `--only <stage_id> --seed {SEED}` if you ever
want it back; `--list` above shows every stage id.

`--time-budget 8.0` stops cleanly before Kaggle's session limit rather than being
killed mid-notebook. `--git-push` commits and pushes `experiments/seed_<SEED>/`
after every stage, so progress survives the session ending.

In [ ]:
%cd /kaggle/working/repo
!python -u scripts/run_pipeline_stages.py --seed {SEED} --time-budget 8.0 --git-push

### After every seed has finished

Once seeds 1, 2 and 3 have all completed (across however many sessions that took),
aggregate them into mean +/- std -- either here or in a local checkout, since it
only reads the committed evaluation tables:

```bash
python scripts/aggregate_seed_results.py
```

Writes `experiments/seed_summary/per_family.csv` (the headline mean/std/n per
family and metric) and `per_cell.csv` (the same, broken down by model year and
eval year). Because the split varies per seed too, the spread reflects both
data-partition and training variance, not training stochasticity alone --
say so when reporting the numbers.